# 19. Cross-year rolling training-window comparison

Compare 30/60/90/180-day training windows with the same 585 regions, forecast origins, 1/2/3-day horizons and weekly refits. Freeze the required-pressure 9-column primary specification and 21-column Backward sensitivity specification from Notebook 18. No feature reselection. Every training target must be dated at or before its weekly fit date. Daily predictions use retrospective onset-date history available in this dataset; reporting delays are not simulated.

Common origins start after the 180-day window plus feature warmup and end three days before source coverage ends. Each year/quarter report excludes origins whose three-day target crosses that period boundary. All windows are scored on identical rows. Calendar quarters are descriptive periods, not formally identified epidemic waves.

2021 was used for feature selection. 2020 is historical replay using a later-selected specification. 2022-2023 outcomes have been examined previously. Therefore this is exploratory cross-year backtesting, not a new untouched test. Later-year training may include earlier evaluation-period observations after their labels mature, as in operational rolling retraining; future labels are never used.


In [1]:
"""Frozen required-pressure specifications: common-date rolling-origin backtest."""
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, sys
import numpy as np
import pandas as pd

def run_cross_year(source, selection_folder, output):
    source,selection_folder=Path(source),Path(selection_folder)
    folder=Path(output)/('cross_year_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    folder.mkdir(parents=True)
    lists=json.loads((selection_folder/'selected_features_by_method.json').read_text())
    specs={'COMPACT_REQUIRED_PRESSURE':lists['FORWARD']['selected_columns'],
           'BACKWARD_REQUIRED_PRESSURE':lists['BACKWARD']['selected_columns']}
    ids=pd.read_csv(selection_folder/'common_regions.csv',dtype={'zip':str})['zip'].to_numpy()
    assert len(ids)==585
    numeric=sorted(set(sum(specs.values(),[]))-set(f'target_weekday_{i}' for i in range(2,8)))
    readcols=[c for c in numeric if c!='no_previous_observed_case']+['zip','date','days_since_last_observed_case','pressure_definition']
    p=pd.read_csv(source,usecols=readcols,dtype={'zip':str},parse_dates=['date'])
    assert not p.duplicated(['zip','date']).any()
    assert p.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    dates=pd.date_range(p.date.min(),p.date.max())
    def matrix(c):return p.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float)
    matrices={c:matrix(c) for c in numeric if c!='no_previous_observed_case'}
    matrices['no_previous_observed_case']=(~np.isfinite(matrix('days_since_last_observed_case'))).astype(float)
    counts=matrices['cases_today'];windows=[30,60,90,180];horizons=[1,2,3]
    # 180 complete training origins, 14 days of feature warmup and h=3 label maturity.
    start=max(pd.Timestamp('2020-07-15'),dates[0]+pd.Timedelta(days=14+180+3))
    origins=np.flatnonzero((dates>=start)&(dates+pd.Timedelta(days=3)<=dates[-1]))
    odates=dates[origins];actual=np.stack([counts[:,origins+h] for h in horizons])
    union=list(dict.fromkeys(sum(specs.values(),[])))
    predictions=np.empty((len(windows),len(specs),3,len(ids),len(origins)))
    timing=[];ranklog=[];coefficients=[];direct_difference=0.
    for hh,h in enumerate(horizons):
        target=dates+pd.Timedelta(days=h)
        for k in range(2,8):matrices[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),counts.shape)
        x=np.stack([matrices[c] for c in union],axis=2)
        for wi,window in enumerate(windows):
            for first in range(0,len(origins),7):
                block=origins[first:first+7];fit=int(block[0]);last=fit-h
                train=np.arange(last-window+1,last+1)
                assert len(train)==window and train[0]>=14 and train[-1]+h<=fit
                xt=x[:,train];yt=counts[:,train+h];xp=x[:,block]
                assert np.isfinite(xt).all() and np.isfinite(xp).all() and np.isfinite(yt).all()
                xm=xt.mean(axis=1);ym=yt.mean(axis=1);xc=xt-xm[:,None];yc=yt-ym[:,None]
                scale=np.sqrt((xc*xc).mean(axis=(0,1)));scale=np.where(scale>1e-10,scale,1)
                z=(xc/scale).reshape(-1,len(union));g=z.T@z;rhs=z.T@yc.ravel()
                timing.append(dict(window=window,horizon=h,fit_date=dates[fit],training_origin_start=dates[train[0]],training_origin_end=dates[train[-1]],latest_training_target=dates[train[-1]+h],forecast_origin_end=dates[block[-1]],training_days=len(train)))
                for mi,(model,features) in enumerate(specs.items()):
                    assert 'nj_covid_pressure_7d' in features
                    ix=[union.index(c) for c in features];keep=[]
                    for j in ix:
                        if g[j,j]<1e-8:continue
                        residual=g[j,j] if not keep else g[j,j]-g[j,keep]@np.linalg.solve(g[np.ix_(keep,keep)],g[keep,j])
                        if residual>1e-8*max(g[j,j],1):keep.append(j)
                    beta=np.linalg.solve(g[np.ix_(keep,keep)],rhs[keep])/scale[keep]
                    alpha=ym-xm[:,keep]@beta
                    pred=np.einsum('ntp,p->nt',xp[:,:,keep],beta)+alpha[:,None]
                    predictions[wi,mi,hh,:,first:first+len(block)]=pred
                    if first==0:
                        direct=np.linalg.lstsq(z[:,keep],yc.ravel(),rcond=None)[0]/scale[keep]
                        dp=np.einsum('ntp,p->nt',xp[:,:,keep],direct)+(ym-xm[:,keep]@direct)[:,None]
                        difference=float(np.max(np.abs(dp-pred)));direct_difference=max(direct_difference,difference)
                        assert np.allclose(dp,pred,atol=1e-7,rtol=1e-7)
                    ranklog.append(dict(window=window,model=model,horizon=h,fit_date=dates[fit],estimated_slopes=len(keep),omitted=';'.join(union[j] for j in ix if j not in keep)))
                    for j,b in zip(keep,beta):coefficients.append(dict(window=window,model=model,horizon=h,fit_date=dates[fit],feature=union[j],coefficient=b))
            print(f'Completed h={h}, window={window}; {len(origins)} common forecast dates.',flush=True)
    assert np.isfinite(predictions).all()
    errors=np.maximum(predictions,0)-actual[None,None]
    ae=np.abs(errors).mean(axis=3);mse=(errors*errors).mean(axis=3);bias=errors.mean(axis=3)
    daily=[]
    for wi,w in enumerate(windows):
        for mi,model in enumerate(specs):
            for hh,h in enumerate(horizons):
                for t,date in enumerate(odates):daily.append(dict(window=w,model=model,horizon=h,forecast_origin=date,MAE=ae[wi,mi,hh,t],MSE=mse[wi,mi,hh,t],bias=bias[wi,mi,hh,t]))
    daily=pd.DataFrame(daily);metrics=[]
    # Require ALL three target dates inside a reporting year/quarter, for identical boundaries.
    groups=[('overall','all_dates',np.ones(len(origins),bool))]
    for year in range(2020,2024):groups.append(('year',str(year),(odates.year==year)&((odates+pd.Timedelta(days=3)).year==year)))
    for quarter in sorted(set(odates.to_period('Q'))):groups.append(('quarter',str(quarter),(odates.to_period('Q')==quarter)&((odates+pd.Timedelta(days=3)).to_period('Q')==quarter)))
    for kind,period,mask in groups:
        if not mask.any():continue
        for wi,w in enumerate(windows):
            for mi,model in enumerate(specs):
                for hh,h in enumerate(horizons):
                    metrics.append(dict(period_type=kind,period=period,window=w,model=model,horizon=h,regions=len(ids),forecast_dates=int(mask.sum()),rows=int(mask.sum()*len(ids)),MAE=float(ae[wi,mi,hh,mask].mean()),RMSE=float(np.sqrt(mse[wi,mi,hh,mask].mean())),bias=float(bias[wi,mi,hh,mask].mean()),mean_actual=float(actual[hh][:,mask].mean())))
    metrics=pd.DataFrame(metrics)
    summary=metrics.groupby(['period_type','period','window','model'],as_index=False)[['MAE','RMSE','bias']].mean()
    summary['MAE_rank_within_period_model']=summary.groupby(['period_type','period','model']).MAE.rank(method='min')
    metrics.to_csv(folder/'metrics_by_period_horizon.csv',index=False);summary.to_csv(folder/'window_summary.csv',index=False)
    daily.to_csv(folder/'daily_losses.csv.gz',index=False)
    pd.DataFrame(timing).to_csv(folder/'training_timing.csv',index=False)
    pd.DataFrame(ranklog).to_csv(folder/'rank_logs.csv',index=False)
    pd.DataFrame(coefficients).to_csv(folder/'rolling_coefficients.csv',index=False)
    pd.DataFrame({'zip':ids}).to_csv(folder/'common_regions.csv',index=False)
    pd.DataFrame({'forecast_origin':odates}).to_csv(folder/'common_forecast_dates.csv',index=False)
    np.savez_compressed(folder/'predictions.npz',raw_predictions=predictions,actual_cases=actual,windows=windows,models=list(specs),horizons=horizons,zip=ids,date=odates.to_numpy())
    protocol=dict(source=str(source),source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),frozen_specs=specs,windows=windows,horizons=horizons,regions=len(ids),refit_every_days=7,forecast_origin_start=str(odates[0].date()),forecast_origin_end=str(odates[-1].date()),information_set='Retrospective onset-date history through forecast origin; latest training label <= weekly fit date. Actual reporting delays not modeled.',scope='Exploratory cross-year backtest, not untouched test. 2021 used for feature selection; 2020 retrospective historical replay; 2022-23 outcomes previously inspected. No feature reselection.',calendar_groups='Year and quarter determined by forecast origin; exclude last three origins crossing group boundary. Calendar quarters are not named epidemic waves.',aggregation='Equal regions per date, equal horizons. Summary RMSE averages horizon-specific RMSE.',feature_selection_source=str(selection_folder))
    (folder/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    checks=dict(all_windows_same_regions_dates_horizons=True,all_predictions_finite=True,all_training_labels_mature_by_fit=True,required_pressure_in_both_specs=True,direct_OLS_max_difference=direct_difference,features_reselected=False,untouched_test_claim=False)
    (folder/'verification.json').write_text(json.dumps(checks,indent=2))
    report='# Cross-year rolling-window backtest\n\nFrozen 9-column primary and 21-column sensitivity specifications, both retaining 7-day Pressure. 585 regions, weekly refits, horizons 1/2/3. Windows share all forecast origins and target values. This is exploratory backtesting; no untouched-test claim.\n\n2020 is partial-year historical replay; 2021 was used for feature selection. Use 2022 and 2023 rows to inspect subsequent temporal stability. Calendar quarters describe time variation, not formally labeled epidemic waves. Lower error is better; do not pick a universal window from one year alone.\n\n## Yearly mean horizon scores\n\n'
    report+=summary.loc[summary.period_type.eq('year')].to_string(index=False)+'\n'
    (folder/'RESULTS.md').write_text(report,encoding='utf-8')
    print(folder.resolve(),flush=True)
    return folder


In [2]:
REPO = Path.cwd()
RUN_FOLDER = REPO / 'cross_year_window_outputs' / 'cross_year_20261026T203232037640Z'
display(pd.read_csv(RUN_FOLDER / 'window_summary.csv'))
print((RUN_FOLDER / 'RESULTS.md').read_text(encoding='utf-8'))


,period_type,period,window,model,MAE,RMSE,bias,MAE_rank_within_period_model
0,overall,all_dates,30,BACKWARD_REQUIRED_PRESSURE,0.782373,1.504398,0.080491,4.0
1,overall,all_dates,30,COMPACT_REQUIRED_PRESSURE,0.778105,1.497406,0.080748,4.0
2,overall,all_dates,60,BACKWARD_REQUIRED_PRESSURE,0.770506,1.473186,0.082843,3.0
3,overall,all_dates,60,COMPACT_REQUIRED_PRESSURE,0.769024,1.477001,0.082128,3.0
4,overall,all_dates,90,BACKWARD_REQUIRED_PRESSURE,0.767244,1.464714,0.073860,2.0
...,...,...,...,...,...,...,...,...
147,year,2023,60,COMPACT_REQUIRED_PRESSURE,0.186091,0.462486,0.026857,1.0
148,year,2023,90,BACKWARD_REQUIRED_PRESSURE,0.186455,0.460181,0.025023,2.0
149,year,2023,90,COMPACT_REQUIRED_PRESSURE,0.188285,0.463517,0.028412,3.0
150,year,2023,180,BACKWARD_REQUIRED_PRESSURE,0.191719,0.465153,0.035773,4.0


# Cross-year rolling-window backtest

Frozen 9-column primary and 21-column sensitivity specifications, both retaining 7-day Pressure. 585 regions, weekly refits, horizons 1/2/3. Windows share all forecast origins and target values. This is exploratory backtesting; no untouched-test claim.

2020 is partial-year historical replay; 2021 was used for feature selection. Use 2022 and 2023 rows to inspect subsequent temporal stability. Calendar quarters describe time variation, not formally labeled epidemic waves. Lower error is better; do not pick a universal window from one year alone.

## Yearly mean horizon scores

period_type period  window                      model      MAE     RMSE     bias  MAE_rank_within_period_model
       year   2020      30 BACKWARD_REQUIRED_PRESSURE 0.911641 1.587169 0.059143                           4.0
       year   2020      30  COMPACT_REQUIRED_PRESSURE 0.906201 1.586529 0.057436                           4.0
       year   2020      60 BACKWARD_REQUIRED_P

## Optional rerun
Skip this cell to inspect the already completed run. Running it creates a separate timestamped output folder.

In [3]:
RUN_FOLDER = run_cross_year(REPO / 'nj_daily_feature_outputs' / 'nj_daily_features.csv.gz', REPO / 'required_pressure_selection_outputs' / 'required_pressure_20261026T183725469346Z', REPO / 'cross_year_window_outputs')


Completed h=1, window=30; 1261 common forecast dates.
Completed h=1, window=60; 1261 common forecast dates.
Completed h=1, window=90; 1261 common forecast dates.
Completed h=1, window=180; 1261 common forecast dates.
Completed h=2, window=30; 1261 common forecast dates.
Completed h=2, window=60; 1261 common forecast dates.
Completed h=2, window=90; 1261 common forecast dates.
Completed h=2, window=180; 1261 common forecast dates.
Completed h=3, window=30; 1261 common forecast dates.
Completed h=3, window=60; 1261 common forecast dates.
Completed h=3, window=90; 1261 common forecast dates.
Completed h=3, window=180; 1261 common forecast dates.
C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\cross_year_window_outputs\cross_year_20261026T204953823819Z


## Files to read

1. `window_summary.csv`: year/quarter and overall MAE/RMSE across horizons; ranks are descriptive.
2. `metrics_by_period_horizon.csv`: scores for horizons 1, 2 and 3 separately.
3. `paired_window_comparisons.csv`: yearly paired date-level MAE differences, pointwise 95% moving-block intervals and HAC p-values, Holm-adjusted across 48 comparisons. Positive differences favor the candidate window. These are exploratory, not post-selection-valid inference.
4. `figures/`: two yearly error comparison figures.
5. `training_timing.csv`, `verification.json`, `protocol.json`: reproducibility and timing checks.
6. `predictions.npz`: all raw predictions, observed targets, common region/date IDs. Clip predictions to zero before scoring, without rounding.

Summary RMSE is the mean of three horizon-specific RMSE values. Do not confuse it with pooled RMSE. Yearly errors reflect differing disease intensity and missing-onset coverage; use within-year matched-window contrasts. No single year establishes a universal best training window.


In [4]:
import math
def hac_mean_test(d,bandwidth=14):
    """Two-sided asymptotic test of zero mean paired DATE-level loss difference."""
    d=np.asarray(d,float)
    assert np.isfinite(d).all()
    T=len(d); L=min(int(bandwidth),T-1)
    mean=float(d.mean()); c=d-mean
    longvar=float(c@c/T)
    for lag in range(1,L+1):
        longvar+=2*(1-lag/(L+1))*float(c[lag:]@c[:-lag]/T)
    variance=longvar/T*T/(T-1)
    if variance<=0 or not np.isfinite(variance):
        return mean,np.nan,np.nan
    se=math.sqrt(variance)
    return mean,se,math.erfc(abs(mean/se)/math.sqrt(2))

def holm_adjust(p):
    p=np.asarray(p,float); adjusted=np.full(p.shape,np.nan)
    good=np.flatnonzero(np.isfinite(p)); ordered=good[np.argsort(p[good])]
    if len(ordered):
        values=np.maximum.accumulate((len(ordered)-np.arange(len(ordered)))*p[ordered])
        adjusted[ordered]=np.minimum(values,1)
    return adjusted

def block_indices(T,block_length,repetitions,seed):
    """Noncircular moving-block resampling; one date index shared by all regions/horizons."""
    assert 1<=block_length<=T
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block_length+1,size=(repetitions,math.ceil(T/block_length)))
    return (starts[:,:,None]+np.arange(block_length)[None,None,:]).reshape(repetitions,-1)[:,:T]

folder = RUN_FOLDER
summary=pd.read_csv(folder/'window_summary.csv');daily=pd.read_csv(folder/'daily_losses.csv.gz',parse_dates=['forecast_origin'])
paired=[]
for model in daily.model.unique():
    for year in range(2020,2024):
        d=daily.loc[daily.model.eq(model)&daily.forecast_origin.dt.year.eq(year)&(daily.forecast_origin+pd.Timedelta(days=3)).dt.year.eq(year)]
        wide=d.groupby(['forecast_origin','window']).MAE.mean().unstack('window')
        assert not wide.isna().any().any()
        draws=block_indices(len(wide),14,2000,20261027)
        for baseline,candidate in itertools.combinations([30,60,90,180],2):
            delta=(wide[baseline]-wide[candidate]).to_numpy();mean,se,p=hac_mean_test(delta,14)
            lo,hi=np.quantile(delta[draws].mean(axis=1),[.025,.975])
            paired.append(dict(model=model,year=year,baseline_window=baseline,candidate_window=candidate,delta_MAE_baseline_minus_candidate=mean,ci95_low_pointwise=lo,ci95_high_pointwise=hi,p_HAC_raw_exploratory=p,dates=len(wide)))
paired=pd.DataFrame(paired);paired['p_Holm_all_48_exploratory']=holm_adjust(paired.p_HAC_raw_exploratory)
paired.to_csv(folder/'paired_window_comparisons.csv',index=False)
timing=pd.read_csv(folder/'training_timing.csv')
assert (pd.to_datetime(timing.latest_training_target)<=pd.to_datetime(timing.fit_date)).all()
assert timing.training_days.eq(timing.window).all()
maxdiff=0
with np.load(folder/'predictions.npz') as a:
    dates=pd.DatetimeIndex(a['date'])
    assert a['raw_predictions'].shape==(4,2,3,585,1261)
    for wi,w in enumerate(a['windows']):
        for mi,model in enumerate(a['models']):
            for hh,h in enumerate(a['horizons']):
                e=np.maximum(a['raw_predictions'][wi,mi,hh],0)-a['actual_cases'][hh]
                for year in range(2020,2024):
                    mask=(dates.year==year)&((dates+pd.Timedelta(days=3)).year==year)
                    ref=pd.read_csv(folder/'metrics_by_period_horizon.csv')
                    r=ref.loc[ref.period_type.eq('year')&ref.period.eq(str(year))&ref.window.eq(w)&ref.model.eq(model)&ref.horizon.eq(h)].iloc[0]
                    maxdiff=max(maxdiff,abs(np.abs(e[:,mask]).mean()-r.MAE),abs(np.sqrt((e[:,mask]**2).mean())-r.RMSE))
assert maxdiff<1e-10
checks=json.loads((folder/'verification.json').read_text());checks.update(saved_yearly_metrics_independently_recomputed=True,max_metric_difference=maxdiff,exact_training_day_counts_verified=True)
(folder/'verification.json').write_text(json.dumps(checks,indent=2))


NameError: name 'itertools' is not defined